In [1]:
#path setup
#since the notebook is in a subfolder, we need to add the src folder to the path
#The issue can be solved by installing the package in editable mode
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Outlier Detection Workflows

This notebook provides two point-level examples:

1. Single raster: visible/NIR bands plus NDVI, SAVI, and NDWI from raster 1.
2. Two rasters: the single-raster features plus red-edge/SWIR bands from raster 2.

Both workflows sample labelled points only and save flagged and cleaned vector outputs.

In [7]:
# Import the point-level extraction and outlier functions
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from ML_LC_Classifier import (
    detect_outliers_per_class,
    remove_outliers,
    save_cleaned_points,
    save_flagged_points,
)

# Point-level outlier detection using two complementary raster inputs
RASTER_PATH_1 = r"C:\AFLICM_Test\Imagery\reswir\merged_sentinel_2025.tif"
RASTER_PATH_2 = r"C:\AFLICM_Test\Imagery\reswir\reswir_mosaic_TL_sentinel2.tif"
POINTS_PATH = r"C:\Users\AFahrezi\Documents\GitHub\Improve_pixel_based_lc_classification\input_data\Sample_V2_update.geojson"
CLASS_FIELD = "ID"
CLASS_NAME_FIELD = "LULC_Type"
ID_FIELD = "point_id"
REMOVE_OUTLIERS = True
metadata_cols = [ID_FIELD, CLASS_FIELD, CLASS_NAME_FIELD]

# These TIFFs have no band descriptions. Confirm both orders match raster creation.
RASTER_1_BAND_ORDER = ["B2", "B3", "B4", "B8"]
RASTER_2_BAND_ORDER = ["B5", "B6", "B11", "B12"]

# McFeeters NDWI: (green - NIR) / (green + NIR).
def extract_raster_features(
    raster_path,
    points_path,
    class_field,
    id_field="point_id",
    class_name_field=None,
    band_order=None,
    calculate_indices=False,
    soil_adjustment=0.5,
):
    points = gpd.read_file(points_path)
    rows = []
    band_order = list(band_order or [])

    with rasterio.open(raster_path) as raster:
        if points.crs is not None and raster.crs is not None and points.crs != raster.crs:
            points = points.to_crs(raster.crs)
        if len(band_order) != raster.count:
            raise ValueError(
                f"{raster_path} has {raster.count} bands, but band_order has "
                f"{len(band_order)} names"
            )
        if len(set(band_order)) != len(band_order):
            raise ValueError("band_order must contain unique band names")

        band_indexes = {
            band_name: index
            for index, band_name in enumerate(band_order, start=1)
        }
        required_bands = {"B3", "B4", "B8"}
        if calculate_indices and not required_bands.issubset(band_indexes):
            raise ValueError(
                f"band_order must include: {sorted(required_bands)}"
            )

        for _, point in points.iterrows():
            if point.geometry is None or point.geometry.is_empty:
                continue
            if point.geometry.geom_type != "Point":
                raise ValueError("Point-level extraction requires Point geometries")
            if pd.isna(point[class_field]) or (
                class_name_field is not None and pd.isna(point[class_name_field])
            ):
                continue

            sampled = next(
                raster.sample(
                    [(point.geometry.x, point.geometry.y)],
                    indexes=list(range(1, raster.count + 1)),
                    masked=True,
                )
            )
            sampled = np.ma.asarray(sampled)
            if np.ma.getmaskarray(sampled).any() or not np.isfinite(sampled.data).all():
                continue

            values = sampled.data.astype(float)
            result = {
                id_field: point[id_field],
                class_field: point[class_field],
                **dict(zip(band_order, values)),
            }
            if calculate_indices:
                green, red, nir = (
                    values[band_indexes[band] - 1]
                    for band in ["B3", "B4", "B8"]
                )
                ndvi_denominator = nir + red
                ndwi_denominator = green + nir
                if ndvi_denominator == 0 or ndwi_denominator == 0:
                    continue
                result.update({
                    "NDVI": (nir - red) / ndvi_denominator,
                    "SAVI": ((nir - red) / (ndvi_denominator + soil_adjustment))
                    * (1 + soil_adjustment),
                    "NDWI": (green - nir) / ndwi_denominator,
                })
            if class_name_field is not None:
                result[class_name_field] = point[class_name_field]
            rows.append(result)

    columns = [id_field, class_field]
    if class_name_field is not None:
        columns.append(class_name_field)
    columns.extend(band_order)
    if calculate_indices:
        columns.extend(["NDVI", "SAVI", "NDWI"])
    return pd.DataFrame(rows, columns=columns)

## Shared setup and feature extraction

The helper below supports both workflows. Band names are configured explicitly because the TIFF files do not contain band descriptions.

In [3]:
# Inspect metadata only; do not read either full raster into memory.
for raster_path, band_order in [
    (RASTER_PATH_1, RASTER_1_BAND_ORDER),
    (RASTER_PATH_2, RASTER_2_BAND_ORDER),
]:
    with rasterio.open(raster_path) as dataset:
        print(f"{raster_path}")
        print(f"  Shape: {dataset.height} rows x {dataset.width} columns")
        print(f"  Bands: {dataset.count}")
        print(f"  Descriptions: {dataset.descriptions}")
        print(f"  Configured order: {band_order}")
        if dataset.count != len(band_order):
            raise ValueError("Configured band order must match raster band count")

C:\AFLICM_Test\Imagery\reswir\merged_sentinel_2025.tif
  Shape: 17350 rows x 38729 columns
  Bands: 4
  Descriptions: (None, None, None, None)
  Configured order: ['B2', 'B3', 'B4', 'B8']
C:\AFLICM_Test\Imagery\reswir\reswir_mosaic_TL_sentinel2.tif
  Shape: 17350 rows x 38729 columns
  Bands: 4
  Descriptions: (None, None, None, None)
  Configured order: ['B5', 'B6', 'B11', 'B12']


## Example 1: single-raster outlier detection

This example uses raster 1 only. It includes the visible/NIR bands and the three indices calculated from those bands.

In [ ]:
single_features = extract_raster_features(
    raster_path=RASTER_PATH_1,
    points_path=POINTS_PATH,
    class_field=CLASS_FIELD,
    id_field=ID_FIELD,
    class_name_field=CLASS_NAME_FIELD,
    band_order=RASTER_1_BAND_ORDER,
    calculate_indices=True,
).rename(
    columns={
        column: f"raster_1_{column}"
        for column in [*RASTER_1_BAND_ORDER, "NDVI", "SAVI", "NDWI"]
    }
)

single_feature_cols = [
    column for column in single_features.columns
    if column not in metadata_cols
]
single_flags = detect_outliers_per_class(
    df=single_features,
    feature_cols=single_feature_cols,
    class_col=CLASS_FIELD,
    id_col=ID_FIELD,
    class_name_col=CLASS_NAME_FIELD,
)
single_flags.to_csv(
    project_root / "output" / "training_point_outlier_flags_single_raster.csv",
    index=False,
)
save_flagged_points(
    points_path=POINTS_PATH,
    flags=single_flags,
    output_path=project_root / "output" / "training_points_flagged_single_raster.shp",
    id_col=ID_FIELD,
)

single_cleaned = (
    remove_outliers(single_features, single_flags, id_col=ID_FIELD)
    if REMOVE_OUTLIERS
    else single_features.copy()
)
single_cleaned.to_csv(
    project_root / "output" / "training_points_cleaned_single_raster.csv",
    index=False,
)
if REMOVE_OUTLIERS:
    save_cleaned_points(
        points_path=POINTS_PATH,
        flags=single_flags,
        output_path=project_root / "output" / "training_points_cleaned_single_raster.shp",
        id_col=ID_FIELD,
    )

print(f"Single-raster samples: {len(single_features)}")
print(f"Single-raster features: {single_feature_cols}")
print(f"Single-raster flagged outliers: {single_flags['outlier'].sum()}")
print("Single-raster flagged vector: output/training_points_flagged_single_raster.shp")

## Example 2: two-raster outlier detection

This example combines raster 1 visible/NIR bands and indices with raster 2 red-edge/SWIR bands. It writes separate outputs from the single-raster example.

In [8]:
two_raster_features_1 = extract_raster_features(
    raster_path=RASTER_PATH_1,
    points_path=POINTS_PATH,
    class_field=CLASS_FIELD,
    id_field=ID_FIELD,
    class_name_field=CLASS_NAME_FIELD,
    band_order=RASTER_1_BAND_ORDER,
    calculate_indices=True,
).rename(
    columns={
        column: f"raster_1_{column}"
        for column in [*RASTER_1_BAND_ORDER, "NDVI", "SAVI", "NDWI"]
    }
)
two_raster_features_2 = extract_raster_features(
    raster_path=RASTER_PATH_2,
    points_path=POINTS_PATH,
    class_field=CLASS_FIELD,
    id_field=ID_FIELD,
    class_name_field=CLASS_NAME_FIELD,
    band_order=RASTER_2_BAND_ORDER,
).rename(
    columns={
        column: f"raster_2_{column}"
        for column in RASTER_2_BAND_ORDER
    }
)

point_features = two_raster_features_1.merge(
    two_raster_features_2,
    on=metadata_cols,
    how="inner",
    validate="one_to_one",
)

feature_cols = [
    column for column in point_features.columns
    if column not in metadata_cols
]
flags = detect_outliers_per_class(
    df=point_features,
    feature_cols=feature_cols,
    class_col=CLASS_FIELD,
    id_col=ID_FIELD,
    class_name_col=CLASS_NAME_FIELD,
)

flags.to_csv(
    project_root / "output" / "training_point_outlier_flags_two_raster.csv",
    index=False,
)
save_flagged_points(
    points_path=POINTS_PATH,
    flags=flags,
    output_path=project_root / "output" / "training_points_flagged_two_raster.shp",
    id_col=ID_FIELD,
)

# Review flags before using the cleaned table for model training.
cleaned_point_features = (
    remove_outliers(point_features, flags, id_col=ID_FIELD)
    if REMOVE_OUTLIERS
    else point_features.copy()
)
cleaned_point_features.to_csv(
    project_root / "output" / "training_cleaned_two_raster.csv",
    index=False,
)

if REMOVE_OUTLIERS:
    save_cleaned_points(
        points_path=POINTS_PATH,
        flags=flags,
        output_path=project_root / "output" / "training_cleaned_two_raster.shp",
        id_col=ID_FIELD,
    )

print(f"Raster 1 samples: {len(two_raster_features_1)}")
print(f"Raster 2 samples: {len(two_raster_features_2)}")
print(f"Shared samples used: {len(point_features)}")
print(f"Two-raster features: {feature_cols}")
print(f"Two-raster flagged outliers: {flags['outlier'].sum()}")
print(f"Samples after filtering: {len(cleaned_point_features)}")
print("Two-raster flagged vector: output/training_flagged_two_raster.shp")

Raster 1 samples: 2820
Raster 2 samples: 2820
Shared samples used: 2820
Two-raster features: ['raster_1_B2', 'raster_1_B3', 'raster_1_B4', 'raster_1_B8', 'raster_1_NDVI', 'raster_1_SAVI', 'raster_1_NDWI', 'raster_2_B5', 'raster_2_B6', 'raster_2_B11', 'raster_2_B12']
Two-raster flagged outliers: 149
Samples after filtering: 2671
Two-raster flagged vector: output/training_flagged_two_raster.shp


c:\Users\AFahrezi\Documents\GitHub\Improve_pixel_based_lc_classification\src\ML_LC_Classifier\outlier_detection.py:231: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  flagged_points.to_file(output_path, driver=driver) # type: ignore
c:\Users\AFahrezi\AppData\Local\anaconda3\envs\ml_lc\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'LULC_Type_flag' to 'LULC_Type_'
  ogr_write(


## Outputs

The single-raster example writes files with the `single_raster` suffix. The two-raster example writes files with the `two_raster` suffix.

Each workflow creates:

- an outlier flag CSV,
- a flagged vector containing all original points and detector results,
- a cleaned CSV, and
- a cleaned vector with flagged points removed.